# Final Combined (no mix\_140\_10) — Data Preparation

Data preparation for the **final, k-fold** evaluation. Unlike the earlier notebook, there is
**no fixed held-out test set** — the previously held-out sessions are folded back in, so
**all 28 sessions** are available. The model notebook will run session-grouped k-fold /
Leave-One-Session-Out so every session is tested.

This notebook only prepares the data:
1. Import the processed MEMS + SPEC pickles (excluding `mix_140_10`).
2. Merge, label, and feature-engineer all sessions.
3. Build the 60\,s windowed feature table `windows_df`.
4. Define `balance_fold()` — bootstrap sweat + undersample baseline — to be applied
   **inside each fold** so cross-validation stays leak-free.

## Data Import

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd

PROCESSED = "processed"

MEMS = ["voc", "nh3", "hcho"]
SPEC = ["h2s_ppm", "etoh_ppm"]
ENV  = ["temp_C", "rh_pct"]
ALL_CHANNELS = MEMS + SPEC + ENV

# Sessions excluded from this model — 6.7% blood is below the NH3 detection floor
EXCLUDE_SESSIONS = ("mix_140_10",)

# ── Auto-discover and load all processed pickles ──────────────────────────────
def _load_pickles(prefix):
    result = {}
    for fname in sorted(os.listdir(PROCESSED)):
        if fname.startswith(prefix) and fname.endswith(".pkl"):
            if any(x in fname for x in EXCLUDE_SESSIONS):
                continue
            key = fname[len(prefix):-4]   # strip prefix and .pkl extension
            result[key] = pd.read_pickle(os.path.join(PROCESSED, fname))
    return result

mems = _load_pickles("mems_")
spec = _load_pickles("spec_")

print(f"MEMS pickles loaded ({len(mems)}):")
for k, df in mems.items():
    print(f"  {k:<12}  {len(df):5} rows  elapsed {df['elapsed_s'].iloc[0]:.0f}–{df['elapsed_s'].iloc[-1]:.0f} s  cols: {list(df.columns)}")

print(f"\nSPEC pickles loaded ({len(spec)}):")
for k, df in spec.items():
    print(f"  {k:<12}  {len(df):5} rows  elapsed {df['elapsed_s'].iloc[0]:.0f}–{df['elapsed_s'].iloc[-1]:.0f} s  cols: {list(df.columns)}")

mems_only = set(mems) - set(spec)
spec_only = set(spec) - set(mems)
if mems_only:
    print(f"\nWARNING: in MEMS but missing SPEC pickle: {sorted(mems_only)}")
if spec_only:
    print(f"\nWARNING: in SPEC but missing MEMS pickle: {sorted(spec_only)}")

MEMS pickles loaded (28):
  blood_0        3666 rows  elapsed 0–3753 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_1        1908 rows  elapsed 1751–3703 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_2        2882 rows  elapsed 0–2987 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_3        3324 rows  elapsed 0–3484 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_4        2357 rows  elapsed 0–2473 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_5        3598 rows  elapsed 0–3746 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_6        3383 rows  elapsed 500–3963 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_7        3486 rows  elapsed 0–3568 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_pct']
  blood_8        3216 rows  elapsed 501–3791 s  cols: ['elapsed_s', 'voc', 'nh3', 'hcho', 'temp_C', 'rh_p

## Merge MEMS + SPEC

In [2]:
MERGE_TOL = 0.5  # seconds — tolerance for nearest-match fallback

datasets = {}

print(f"{'Dataset':<12}  {'MEMS':>6}  {'SPEC':>6}  {'Merged':>7}  {'Dropped':>8}  {'Method'}")
print("-" * 58)

for key in sorted(mems):
    if key not in spec:
        print(f"{key:<12}  -- no SPEC pickle, skipped --")
        continue

    m = mems[key]
    s = spec[key]

    # Exact merge on elapsed_s
    exact = pd.merge(
        m,
        s[["elapsed_s"] + SPEC],
        on="elapsed_s",
        how="inner",
    )
    n_exact_dropped = max(len(m), len(s)) - len(exact)

    if n_exact_dropped == 0:
        merged = exact
        method = "exact"
    else:
        # Nearest-match fallback — sorts required by merge_asof
        m_s = m.sort_values("elapsed_s").reset_index(drop=True)
        s_s = s[["elapsed_s"] + SPEC].sort_values("elapsed_s").reset_index(drop=True)
        merged = pd.merge_asof(
            m_s, s_s,
            on="elapsed_s",
            tolerance=MERGE_TOL,
            direction="nearest",
        )
        merged = merged.dropna(subset=SPEC).reset_index(drop=True)
        method = f"nearest (tol={MERGE_TOL}s)"

    n_dropped = max(len(m), len(s)) - len(merged)
    flag = "  *** ROWS DROPPED ***" if n_dropped > 0 else ""

    merged = merged[["elapsed_s"] + ALL_CHANNELS].reset_index(drop=True)
    merged["session"] = key
    datasets[key] = merged

    print(f"{key:<12}  {len(m):>6}  {len(s):>6}  {len(merged):>7}  {n_dropped:>8}  {method}{flag}")

print(f"\nTotal sessions merged: {len(datasets)}")

Dataset         MEMS    SPEC   Merged   Dropped  Method
----------------------------------------------------------
blood_0         3666    3666     3666         0  exact
blood_1         1908    1908     1908         0  exact
blood_2         2882    2882     2882         0  exact
blood_3         3324    3324     3324         0  exact
blood_4         2357    2357     2357         0  exact
blood_5         3598    3598     3598         0  exact
blood_6         3383    3383     3383         0  exact
blood_7         3486    3486     3486         0  exact
blood_8         3216    3216     3216         0  exact
blood_9         4093    4093     4093         0  exact
mix_120_30_1    2857    2857     2857         0  exact
mix_120_30_2    3169    3169     3169         0  exact
mix_120_30_3    2866    2866     2866         0  exact
mix_120_30_4    2638    2638     2638         0  exact
mix_120_30_5    2798    2798     2798         0  exact
mix_75_75_1     6184    6184     6184         0  exact
mix_7

## Data Labeling

All 28 sessions are labeled here — there is no separate held-out test set.

In [3]:
LABEL_BASELINE = 0
LABEL_SWEAT    = 1
LABEL_BLOOD    = 2

# (t_start, t_end, label) — t_end=None means end of session
SAMPLE_WINDOWS = {
    "sweat_1a":     [(2870, 3860,  LABEL_SWEAT)],
    "sweat_1b":     [(570,  2330,  LABEL_SWEAT)],
    "sweat_2":      [(4220, 5460,  LABEL_SWEAT)],
    "sweat_3":      [(3550, 5227,  LABEL_SWEAT)],
    "sweat_4":      [(3860, 5042,  LABEL_SWEAT)],
    "sweat_5":      [(1140, 2786,  LABEL_SWEAT)],
    "sweat_6":      [(4600, 6120,  LABEL_SWEAT)],
    "sweat_7":      [(1760, 3610,  LABEL_SWEAT)],
    "blood_0":      [(1900, 3730,  LABEL_BLOOD)],
    "blood_1":      [(2440, 3703,  LABEL_BLOOD)],
    "blood_2":      [(1770, 2987,  LABEL_BLOOD)],
    "blood_3":      [(1950, 3484,  LABEL_BLOOD)],
    "blood_4":      [(1350, 2440,  LABEL_BLOOD)],
    "blood_5":      [(1750, 3746,  LABEL_BLOOD)],
    "blood_6":      [(2650, 3950,  LABEL_BLOOD)],
    "blood_7":      [(2150, 3568,  LABEL_BLOOD)],
    "blood_8":      [(2430, 3791,  LABEL_BLOOD)],
    "blood_9":      [(2290, 4190,  LABEL_BLOOD)],
    "mix_75_75_1":  [(4300, 6700,  LABEL_BLOOD)],
    "mix_75_75_2":  [(3910, 5650,  LABEL_BLOOD)],
    "mix_75_75_3":  [(1390, None,  LABEL_BLOOD)],
    "mix_75_75_4":  [(1600, None,  LABEL_BLOOD)],
    "mix_75_75_5":  [(2360, None,  LABEL_BLOOD)],
    "mix_120_30_1": [(1350, None,  LABEL_BLOOD)],
    "mix_120_30_2": [(1625, None,  LABEL_BLOOD)],
    "mix_120_30_3": [(1580, 2880,  LABEL_BLOOD)],
    "mix_120_30_4": [(1675, None,  LABEL_BLOOD)],
    "mix_120_30_5": [(1340, None,  LABEL_BLOOD)],
    "mix_140_10_1": [(1645, 3260,  LABEL_BLOOD)],
    "mix_140_10_2": [(6050, None,  LABEL_BLOOD)],
    "mix_140_10_3": [(5925, None,  LABEL_BLOOD)],
    "mix_140_10_4": [(4825, None,  LABEL_BLOOD)],
    "mix_140_10_5": [(4950, None,  LABEL_BLOOD)],
}

for key, df in datasets.items():
    datasets[key] = df.copy()
    datasets[key]["label"] = LABEL_BASELINE
    for t_start, t_end, lbl in SAMPLE_WINDOWS.get(key, []):
        t = df["elapsed_s"]
        mask = (t >= t_start) & (t <= (t_end if t_end is not None else t.iloc[-1]))
        datasets[key].loc[mask, "label"] = lbl

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"{'Dataset':<14}  {'baseline':>10}  {'sweat':>8}  {'blood':>8}  {'total':>8}  {'sample %':>9}")
print("-" * 66)
for key, df in sorted(datasets.items()):
    b  = (df["label"] == LABEL_BASELINE).sum()
    sw = (df["label"] == LABEL_SWEAT).sum()
    bl = (df["label"] == LABEL_BLOOD).sum()
    sample_pct = 100 * (sw + bl) / len(df)
    print(f"{key:<14}  {b:>10}  {sw:>8}  {bl:>8}  {len(df):>8}  {sample_pct:>8.1f}%")


Dataset           baseline     sweat     blood     total   sample %
------------------------------------------------------------------
blood_0               1879         0      1787      3666      48.7%
blood_1                675         0      1233      1908      64.6%
blood_2               1717         0      1165      2882      40.4%
blood_3               1860         0      1464      3324      44.0%
blood_4               1316         0      1041      2357      44.2%
blood_5               1690         0      1908      3598      53.0%
blood_6               2113         0      1270      3383      37.5%
blood_7               2101         0      1385      3486      39.7%
blood_8               1887         0      1329      3216      41.3%
blood_9               2237         0      1856      4093      45.3%
mix_120_30_1          1319         0      1538      2857      53.8%
mix_120_30_2          1548         0      1621      3169      51.2%
mix_120_30_3          1597         0      1269   

### Label Visualization

Shaded regions show the labelled time windows — **grey** = baseline, **green** = sweat, **red** = blood.

In [ ]:
import matplotlib.patches as mpatches

REGION_COLORS = {
    LABEL_BASELINE: ("lightgrey",   "Baseline"),
    LABEL_SWEAT:    ("lightgreen",  "Sweat"),
    LABEL_BLOOD:    ("lightsalmon", "Blood"),
}

PLOT_COLS = ["voc", "nh3", "hcho", "h2s_ppm", "etoh_ppm"]

for key, df in sorted(datasets.items()):
    fig, axes = plt.subplots(len(PLOT_COLS), 1, figsize=(13, 9), sharex=True)
    fig.suptitle(f"{key} — Labelled regions", fontsize=10, fontweight="bold")

    # Build contiguous label blocks for shading
    label_blocks = df[["elapsed_s", "label"]].copy()
    label_blocks["block"] = (label_blocks["label"] != label_blocks["label"].shift()).cumsum()
    for _, grp in label_blocks.groupby("block"):
        t0  = grp["elapsed_s"].iloc[0]
        t1  = grp["elapsed_s"].iloc[-1]
        lbl = grp["label"].iloc[0]
        color, _ = REGION_COLORS[lbl]
        for ax in axes:
            ax.axvspan(t0, t1, color=color, alpha=0.35, linewidth=0)

    for ax, col in zip(axes, PLOT_COLS):
        ax.plot(df["elapsed_s"], df[col], linewidth=0.8, color="black", alpha=0.8)
        ax.set_ylabel(col, fontsize=8)
        ax.grid(True, alpha=0.25, linestyle="--")

    axes[-1].set_xlabel("elapsed (s)", fontsize=8)

    patches = [mpatches.Patch(color=c, label=l, alpha=0.5)
               for _, (c, l) in REGION_COLORS.items()]
    axes[0].legend(handles=patches, fontsize=7, loc="upper left")

    plt.tight_layout()
    plt.show()

## Feature Engineering

In [4]:
import numpy as np

CHANNELS     = ["voc", "nh3", "hcho", "h2s_ppm", "etoh_ppm", "rh_pct"]
ROLL_WINDOWS = [15, 30, 60]   # seconds at 1 Hz

EXCLUDE_COLS = {"elapsed_s", "label", "session", "temp_C"}

for key in datasets:
    df = datasets[key].copy()

    for col in CHANNELS:
        df[f"{col}_roc"] = df[col].diff()
        df[f"{col}_acc"] = df[f"{col}_roc"].diff()
        for w in ROLL_WINDOWS:
            df[f"{col}_roll_mean_{w}"] = df[col].rolling(window=w, min_periods=1).mean()
            df[f"{col}_roll_std_{w}"]  = df[col].rolling(window=w, min_periods=1).std().fillna(0)
            df[f"{col}_roll_roc_{w}"]  = df[f"{col}_roc"].abs().rolling(window=w, min_periods=1).mean()

    datasets[key] = df

# FEATURE_COLS defined here — used by downstream modelling cells
_sample = next(iter(datasets.values()))
FEATURE_COLS = [c for c in _sample.columns if c not in EXCLUDE_COLS]

helper_cols = [c for c in FEATURE_COLS if c not in set(ALL_CHANNELS)]
print(f"Sessions processed : {len(datasets)}")
print(f"Helper cols added  : {len(helper_cols)}  per session")
print(f"Total feature cols : {len(FEATURE_COLS)}  (signal + helpers, excl. elapsed/label/temp)")
print(f"\nColumns per session: {len(_sample.columns)}  total")


Sessions processed : 28
Helper cols added  : 66  per session
Total feature cols : 72  (signal + helpers, excl. elapsed/label/temp)

Columns per session: 76  total


## Windowed Feature Extraction

A 60 s **non-overlapping** window is collapsed to one row of aggregate features
(mean, std, max, slope per feature column). `windows_df` keeps a `session` column for
session-grouped cross-validation.

**Why non-overlapping?** A sliding/overlapping window would emit a new row every few
seconds, and consecutive windows would share almost all of their samples — producing many
near-duplicate, highly-correlated rows. That repetitive data inflates the apparent sample
size, lets the model see essentially the same window many times (encouraging memorization),
and makes correlated copies easy to leak across a train/test boundary. Non-overlapping
windows give cleaner, near-independent training examples — each second of signal
contributes to exactly one window.

In [5]:
WINDOW_SIZE = 60   # seconds = samples at 1 Hz

# FEATURE_COLS is defined in the Feature Engineering cell above
def extract_window_features(window_df, feature_cols):
    features = {}
    t = np.arange(len(window_df))
    for col in feature_cols:
        vals = window_df[col].values
        features[f"{col}_mean"]  = np.nanmean(vals)
        features[f"{col}_std"]   = np.nanstd(vals)
        features[f"{col}_max"]   = np.nanmax(vals)
        slope, _ = np.polyfit(t, vals, 1) if len(vals) > 1 else (0.0, 0.0)
        features[f"{col}_slope"] = slope
    return features

# ── Slide window per (session, label) — never cross label boundaries ──────────
# Mix sessions carry LABEL_BLOOD, so their windows land in the blood class and are
# kept through to training — nothing here discards them.
all_windows = []
for session, df in datasets.items():
    for label in [LABEL_BASELINE, LABEL_SWEAT, LABEL_BLOOD]:
        label_df  = df[df["label"] == label].reset_index(drop=True)
        n_windows = len(label_df) // WINDOW_SIZE
        for i in range(n_windows):
            window = label_df.iloc[i * WINDOW_SIZE : (i + 1) * WINDOW_SIZE]
            feats  = extract_window_features(window, FEATURE_COLS)
            feats["label"]   = label
            feats["session"] = session
            all_windows.append(feats)

windows_df = pd.DataFrame(all_windows)

print(f"Built windows_df: {len(windows_df)} windows from {windows_df['session'].nunique()} sessions.")
print(f"Features per window: {len(FEATURE_COLS) * 4}  (×4 aggregates)")
print("Balancing + bootstrap happen in the next section.")

Built windows_df: 1683 windows from 28 sessions.
Features per window: 288  (×4 aggregates)
Balancing + bootstrap happen in the next section.


## Sweat Bootstrap / Per-Fold Balancing

`balance_fold()` balances the three classes for one fold's **training** windows:

- **Bootstrap sweat** up to the blood count by duplicating sweat windows with small
  Gaussian noise (`x + N(0, \sigma^2)`, `\sigma = noise\_scale \times \text{std}`).
- **Undersample baseline** down to the same count.

**Apply it inside each k-fold split, on the training windows only** — never on the full
`windows_df` before splitting — or synthetic copies of a held-out session would leak into
training and inflate the score.

In [6]:
def balance_fold(train_df, feature_cols, noise_scale=0.05, random_state=42):
    """Balance one CV fold's TRAINING windows: bootstrap sweat up to the blood count
    (with Gaussian noise), then undersample baseline to match. Call per fold only."""
    rng   = np.random.default_rng(random_state)
    sweat = train_df[train_df["label"] == LABEL_SWEAT]
    blood = train_df[train_df["label"] == LABEL_BLOOD]
    base  = train_df[train_df["label"] == LABEL_BASELINE]

    # bootstrap sweat -> blood count
    n_add = len(blood) - len(sweat)
    if n_add > 0 and len(sweat) > 0:
        stds = sweat[feature_cols].std().fillna(0).values
        syn  = sweat.sample(n=n_add, replace=True, random_state=random_state).copy()
        syn[feature_cols] = syn[feature_cols].values + rng.normal(0, noise_scale, size=(n_add, len(feature_cols))) * stds
        syn["session"] = "bootstrap"
        sweat = pd.concat([sweat, syn], ignore_index=True)

    # undersample baseline to the target
    target = max(len(sweat), len(blood))
    base_s = base.sample(n=min(target, len(base)), random_state=random_state)

    return pd.concat([sweat, blood, base_s]).reset_index(drop=True)

# Illustration on the full set (in CV, call this per fold on the fold's training rows)
_feat = [c for c in windows_df.columns if c not in ("label", "session")]
_demo = balance_fold(windows_df, _feat)
print("Class counts — full set, before vs after balance (illustration only):")
for lbl, name in [(LABEL_BASELINE, "baseline"), (LABEL_SWEAT, "sweat"), (LABEL_BLOOD, "blood")]:
    print(f"  {name:<9}  before {(windows_df['label']==lbl).sum():>4}   after {(_demo['label']==lbl).sum():>4}")
print(f"  total      before {len(windows_df):>4}   after {len(_demo):>4}")

Class counts — full set, before vs after balance (illustration only):
  baseline   before  981   after  513
  sweat      before  189   after  513
  blood      before  513   after  513
  total      before 1683   after 1539
